# Data Pipeline Verification Sandbox

This notebook now performs two roles:

1. Build the processed DEMADICS `.npy` dataset from the untouched raw DAMADICS archives.
2. Verify every active FlowMatch-PdM data pipeline with one real training batch.

DEMADICS sources used for the preprocessing contract:
- Local description manual: `datasets/Damadics/damadics-lublin-data-description-v02March2002.pdf`
- Local benchmark document: `datasets/Damadics/damadics-benchmark-definition.zip` -> `damadics-benchmark-definition-v1_0.pdf`
- Research context: Step III real-process benchmark papers describe 25 daily files, 4 faulty days, and 19 labeled artificial faults grouped into `f16`, `f17`, `f18`, and `f19`.

FlowMatch-PdM DEMADICS contract used here:
- task: 5-class classification (`normal`, `f16`, `f17`, `f18`, `f19`)
- input: all 32 non-timestamp process variables
- window size: 2048
- split: strict stratified 80/10/10
- scaling: z-score fit on train split only
- fault windows: event-centered sampling from the labeled fault intervals, capped per event so the long open-ended `f17` case does not dominate the dataset


In [ ]:
from pathlib import Path
import sys

import torch
import yaml

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "configs" / "default_config.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.data_helper import get_data_module, get_dataset_config
from src.utils.demadics_preprocessing import (
    DEMADICS_CLASS_TO_INDEX,
    DEMADICS_FAULT_EVENTS,
    demadics_paths,
    prepare_demadics_processed,
)

CONFIG = yaml.safe_load((REPO_ROOT / "configs" / "default_config.yaml").read_text())
BATCH_SIZE = 8

print(f"Repo root: {REPO_ROOT}")
print(f"Torch: {torch.__version__}")


In [ ]:
FORCE_REBUILD_DEMADICS = False

paths = demadics_paths(REPO_ROOT)
metadata = prepare_demadics_processed(REPO_ROOT, force_rebuild=FORCE_REBUILD_DEMADICS)

print("DEMADICS preprocessing ready")
print(f"Archive dir:   {paths['archive_dir']}")
print(f"Raw dir:       {metadata['raw_dir']}")
print(f"Processed dir: {metadata['processed_dir']}")
print(f"Fault events encoded from manual: {len(DEMADICS_FAULT_EVENTS)}")
print(f"Class map: {DEMADICS_CLASS_TO_INDEX}")
print(f"Split sizes -> train: {metadata['train_size']}, val: {metadata['val_size']}, test: {metadata['test_size']}")
print(f"Train class counts: {metadata['train_class_counts']}")
print(f"Val class counts:   {metadata['val_class_counts']}")
print(f"Test class counts:  {metadata['test_class_counts']}")


In [ ]:
SUPPORTED_DATASETS = [
    {"dataset": "CMAPSS", "track": "engine_rul", "fd": 1},
    {"dataset": "N-CMAPSS", "track": "engine_rul", "fd": 1},
    {"dataset": "FEMTO", "track": "bearing_rul", "fd": 1},
    {"dataset": "XJTU-SY", "track": "bearing_rul", "fd": 1},
    {"dataset": "CWRU", "track": "bearing_fault"},
    {"dataset": "Paderborn", "track": "bearing_fault"},
    {"dataset": "DEMADICS", "track": "bearing_fault"},
]

def build_datamodule(spec: dict, batch_size: int = BATCH_SIZE):
    dataset_cfg = get_dataset_config(CONFIG, spec["dataset"])
    kwargs = {
        "track": spec["track"],
        "dataset_name": spec["dataset"],
        "window_size": dataset_cfg["window_size"],
        "batch_size": batch_size,
    }
    if "fd" in spec:
        kwargs["fd"] = spec["fd"]
    return get_data_module(**kwargs)

def assert_batch_window_first(x_batch: torch.Tensor, expected_window: int, dataset_name: str) -> None:
    assert x_batch.ndim == 3, f"{dataset_name}: expected a 3D batch, got shape {tuple(x_batch.shape)}"
    assert x_batch.shape[1] == expected_window, f"{dataset_name}: expected [batch, {expected_window}, features], got {tuple(x_batch.shape)}"
    assert x_batch.shape[2] > 0, f"{dataset_name}: feature dimension must be positive"

def assert_target_dtype(y_batch: torch.Tensor, track: str, dataset_name: str) -> None:
    if "rul" in track:
        assert torch.is_floating_point(y_batch), f"{dataset_name}: expected floating-point RUL targets, got {y_batch.dtype}"
        return
    integer_dtypes = {torch.int8, torch.int16, torch.int32, torch.int64, torch.uint8}
    assert y_batch.dtype in integer_dtypes, f"{dataset_name}: expected integer classification targets, got {y_batch.dtype}"


In [ ]:
results = []

for spec in SUPPORTED_DATASETS:
    dataset_name = spec["dataset"]
    dataset_cfg = get_dataset_config(CONFIG, dataset_name)
    print("=" * 96)
    print(f"Dataset Name: {dataset_name}")
    try:
        dm = build_datamodule(spec)
        dm.prepare_data()
        dm.setup("fit")
        x_batch, y_batch = next(iter(dm.train_dataloader()))
        assert_batch_window_first(x_batch, dataset_cfg["window_size"], dataset_name)
        assert_target_dtype(y_batch, spec["track"], dataset_name)
        minority_ds = dm.get_minority_dataset()
        print(f"X_train_batch.shape: {tuple(x_batch.shape)}")
        print(f"y_train_batch.shape: {tuple(y_batch.shape)}")
        print(f"y_train_batch.dtype: {y_batch.dtype}")
        print(f"Minority subset size: {len(minority_ds)} / {len(dm.train_ds)}")
        results.append({"dataset": dataset_name, "status": "PASS", "message": ""})
    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        print(f"Verification failed: {message}")
        results.append({"dataset": dataset_name, "status": "FAIL", "message": message})

print("\n" + "=" * 96)
print("Verification Summary")
for result in results:
    suffix = f" ({result['message']})" if result["message"] else ""
    print(f"- {result['dataset']}: {result['status']}{suffix}")
supported_failures = [result for result in results if result["status"] != "PASS"]
print(f"\nSupported loader readiness: {'GO' if not supported_failures else 'NO-GO'}")
print(f"Full requested roster readiness: {'GO' if not supported_failures else 'NO-GO'}")
